# NB_patient_360 — Gold Layer

Builds `adb_caresync.gold.patient_360`: one row per patient enriched with hospital, insurance provider, and aggregated lab-result metrics. Full rebuild every run.

In [0]:
%sql
-- ============================================================
-- gold.patient_360
-- Grain : one row per patient
-- Load  : FULL — CREATE OR REPLACE TABLE ... AS SELECT
-- Sources: silver.patients (driving)
--          silver.hospitals        (LEFT JOIN on registered_hospital_id)
--          silver.insurance_providers (LEFT JOIN on insurance_provider_id)
--          silver.lab_results      (aggregated subquery by patient_id)
-- ============================================================

CREATE OR REPLACE TABLE adb_caresync.gold.patient_360 AS

WITH lab_agg AS (
  SELECT
    patient_id,
    COUNT(*)                                                   AS total_tests,
    COUNT(*) FILTER (WHERE result_status = 'Abnormal')         AS abnormal_test_count,
    COUNT(*) FILTER (WHERE result_status = 'Critical')         AS critical_test_count,
    MAX(test_date)                                             AS last_test_date
  FROM adb_caresync.silver.lab_results
  GROUP BY patient_id
)

SELECT
  -- Keys & identifiers
  p.patient_id,
  p.first_name,
  p.last_name,

  -- Demographics
  CAST(p.date_of_birth AS DATE)                                                  AS date_of_birth,
  CAST(FLOOR(DATEDIFF(CURRENT_DATE(), CAST(p.date_of_birth AS DATE)) / 365.25) AS INT)  AS age,
  p.gender,
  p.city,
  p.state,

  -- Hospital
  p.registered_hospital_id                                                       AS hospital_id,
  h.hospital_name,

  -- Insurance
  p.insurance_provider_id,
  i.provider_name                                                                AS insurance_provider_name,

  -- Lab result aggregates (0 when no tests exist)
  COALESCE(l.total_tests,          0)                                            AS total_tests,
  COALESCE(l.abnormal_test_count,  0)                                            AS abnormal_test_count,
  COALESCE(l.critical_test_count,  0)                                            AS critical_test_count,
  l.last_test_date

FROM adb_caresync.silver.patients p
LEFT JOIN adb_caresync.silver.hospitals h
  ON p.registered_hospital_id = h.hospital_id
LEFT JOIN adb_caresync.silver.insurance_providers i
  ON p.insurance_provider_id = i.insurance_provider_id
LEFT JOIN lab_agg l
  ON p.patient_id = l.patient_id

In [0]:
record_count = spark.table("adb_caresync.gold.patient_360").count()
dbutils.notebook.exit(str(record_count))